# CSV - Schema Mismatch

Rename or remove required columns so the saved CSV no longer matches its expected schema.

**Instructions:** Use `prepare_data_csv_update.ipynb` to create a clean `data/weather.csv` first. This notebook overwrites that CSV. The analyst must have an expected column contract (e.g., in the exercise README); otherwise a changed header is not demonstrably a defect. Restore the clean source before rerunning.


In [ ]:
from pathlib import Path
from random import Random
import pandas as pd

# Configuration
DATA_PATH = Path("../../data/weather.csv")
SEED = 814205

# Schema Mismatch
SCHEMA_CANDIDATES = ["temperature_2m", "relative_humidity_2m"]
SCHEMA_KIND = "mixed"  # "renamed", "removed", "mixed", "random".
SCHEMA_N_FIELDS = 2  # Number of distinct required columns to change (not a row percentage).
RENAME_SUFFIX = "_source"  # Example: temperature_2m -> temperature_2m_source.


In [ ]:
def add_schema_mismatch(df, rng, *, candidate_columns, kind, n_fields, rename_suffix):
    # Only required fields belong in candidate_columns; the expected schema is external.
    if not candidate_columns or len(candidate_columns) != len(set(candidate_columns)):
        raise ValueError("Provide distinct required candidate columns.")
    if any(col not in df.columns for col in candidate_columns):
        raise ValueError("A required candidate column is already absent. Restore clean input first.")
    if kind not in ("renamed", "removed", "mixed", "random"):
        raise ValueError("Unrecognized schema mismatch kind.")
    if not isinstance(n_fields, int) or not 1 <= n_fields <= len(candidate_columns):
        raise ValueError("n_fields must be between 1 and the number of candidate columns.")
    if not rename_suffix:
        raise ValueError("Rename suffix must be nonempty.")

    allowed = ("renamed", "removed", "mixed") if n_fields >= 2 else ("renamed", "removed")
    selected_kind = rng.choice(allowed) if kind == "random" else kind
    if selected_kind == "mixed" and n_fields < 2:
        raise ValueError("Mixed schema mismatch needs at least two fields.")

    chosen = rng.sample(list(candidate_columns), k=n_fields)
    if selected_kind == "mixed":
        changes = ["renamed", "removed"] + [rng.choice(("renamed", "removed")) for _ in range(n_fields - 2)]
        rng.shuffle(changes)
    else:
        changes = [selected_kind] * n_fields

    rename_map = {col: col + rename_suffix for col, change in zip(chosen, changes) if change == "renamed"}
    removed = [col for col, change in zip(chosen, changes) if change == "removed"]
    if len(removed) == len(df.columns):
        raise ValueError("Cannot remove every CSV column; no readable header would remain.")
    if len(set(rename_map.values())) != len(rename_map) or any(new in df.columns for new in rename_map.values()):
        raise ValueError("Renamed column would collide with an existing column.")

    dirty = df.copy()
    if rename_map:
        dirty = dirty.rename(columns=rename_map)
    if removed:
        dirty = dirty.drop(columns=removed)
    if any(col in dirty.columns for col in chosen):
        raise AssertionError("A selected required column was not changed.")
    if not all(new in dirty.columns for new in rename_map.values()):
        raise AssertionError("Renamed headers were not retained.")
    if len(dirty) != len(df):
        raise AssertionError("A schema mutation must not change the record count.")
    return dirty


In [ ]:
# Small business-like trial: order data has an explicitly required contract.
orders = pd.DataFrame({
    "order_id": [101, 102, 103],
    "order_total": [50.0, 75.0, 20.0],
    "customer_id": [7, 8, 7],
    "channel": ["online", "retail", "online"],
})
trial = add_schema_mismatch(
    orders, Random(SEED), candidate_columns=["order_total", "customer_id"],
    kind="mixed", n_fields=2, rename_suffix="_source",
)
assert len(trial) == len(orders)
assert {"order_total", "customer_id"}.isdisjoint(trial.columns)
assert len([c for c in trial.columns if c.endswith("_source")]) == 1
pd.testing.assert_frame_equal(trial[["order_id", "channel"]], orders[["order_id", "channel"]])
trial.head()


In [ ]:
# Apply to a freshly prepared CSV; overwrite weather.csv.
clean = pd.read_csv(DATA_PATH)
expected_columns = clean.columns.tolist()  # Developer-side expected contract.
dirty = add_schema_mismatch(
    clean, Random(SEED), candidate_columns=SCHEMA_CANDIDATES,
    kind=SCHEMA_KIND, n_fields=SCHEMA_N_FIELDS, rename_suffix=RENAME_SUFFIX,
)
dirty.to_csv(DATA_PATH, index=False)
saved = pd.read_csv(DATA_PATH)


In [ ]:
# Inspect the saved schema (rather than looking for nulls).
saved.columns.tolist()


In [ ]:
# What required fields are missing from the expected contract?
missing_required = [col for col in expected_columns if col not in saved.columns]
new_fields = [col for col in saved.columns if col not in expected_columns]
missing_required, new_fields


In [ ]:
# Verify unchanged rows/values under preserved or renamed columns.
assert len(saved) == len(clean)
assert len(missing_required) == SCHEMA_N_FIELDS
assert set(missing_required).issubset(SCHEMA_CANDIDATES)
assert len(new_fields) <= SCHEMA_N_FIELDS

unchanged = [col for col in expected_columns if col in saved.columns]
pd.testing.assert_frame_equal(
    saved[unchanged], clean[unchanged], check_exact=False, rtol=1e-12,
)
renamed = {col: col + RENAME_SUFFIX for col in missing_required if col + RENAME_SUFFIX in saved.columns}
for old, new in renamed.items():
    pd.testing.assert_series_equal(
        saved[new], clean[old], check_names=False, check_exact=False, rtol=1e-12,
    )
print("Required columns absent:", missing_required)
print("Renamed columns:", renamed)
print("Removed columns:", [col for col in missing_required if col not in renamed])


In [ ]:
# Developer-side diagnostic table; useful as a post-exercise README figure/table.
contract_view = pd.DataFrame({
    "expected_column": expected_columns,
    "present": [col in saved.columns for col in expected_columns],
    "renamed_as": [col + RENAME_SUFFIX if col + RENAME_SUFFIX in saved.columns else "" for col in expected_columns],
})
contract_view
